In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
llm.invoke("How will the weather be in munich today?")

AIMessage(content="I'm sorry, but I can't provide real-time weather updates. I recommend checking a reliable weather website or a weather app for the most current information on the weather in Munich today.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 17, 'total_tokens': 52, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ebf4e532f9', 'id': 'chatcmpl-DQH0qBc7hNoQjihr1cG9MwkFt7zDp', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--b787da8f-0911-4dab-9c8f-aa4810178de5-0', usage_metadata={'input_tokens': 17, 'output_tokens': 35, 'total_tokens': 52, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [4]:
from langchain_core.tools import tool


@tool
def fake_weather_api(city: str) -> str:
    """
    Check the weather in a specified city.

    Args:
        city (str): The name of the city where you want to check the weather.

    Returns:
        str: A description of the current weather in the specified city.
    """
    return "Sunny, 22°C"


tools = [fake_weather_api]

In [5]:
llm_with_tools = llm.bind_tools(tools)

In [6]:
result = llm_with_tools.invoke("How will the weather be in munich today?")
result

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_BfG1GVbpEVRYvvfWEgza9SPU', 'function': {'arguments': '{"city":"Munich"}', 'name': 'fake_weather_api'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 90, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DQH1FIHSBGbTQdujzl7VuvpRdgDCr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--9f18e92e-7848-41bf-aa09-5010b20104e7-0', tool_calls=[{'name': 'fake_weather_api', 'args': {'city': 'Munich'}, 'id': 'call_BfG1GVbpEVRYvvfWEgza9SPU', 'type': 'tool_call'}], usage_metadata={'input_tokens': 90, 'output_tokens': 16, 'total_tokens': 106, 'input_token_deta

In [7]:
result.tool_calls

[{'name': 'fake_weather_api',
  'args': {'city': 'Munich'},
  'id': 'call_BfG1GVbpEVRYvvfWEgza9SPU',
  'type': 'tool_call'}]

In [8]:
from langchain_core.messages import HumanMessage, ToolMessage

messages = [
    HumanMessage(
        "How will the weather be in munich today?"
    )
]
llm_output = llm_with_tools.invoke(messages)
messages.append(llm_output)

In [9]:
messages

[HumanMessage(content='How will the weather be in munich today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_zxwUrmJCcaPFTWAumbfy6IJZ', 'function': {'arguments': '{"city":"Munich"}', 'name': 'fake_weather_api'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 90, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DQH1apwMtNYAuqIND8kMO8DfHpdFz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--77c5263e-ea49-4446-9347-412ae4181796-0', tool_calls=[{'name': 'fake_weather_api', 'args': {'city': 'Munich'}, 'id': 'call_zxwUrmJCcaPFTWAumbfy6IJZ', 'type'

In [10]:
tool_mapping = {"fake_weather_api": fake_weather_api}

In [11]:
for tool_call in llm_output.tool_calls:
    tool = tool_mapping[tool_call["name"].lower()]
    tool_output = tool.invoke(tool_call["args"])
    messages.append(ToolMessage(tool_output, tool_call_id=tool_call["id"]))

In [12]:
messages

[HumanMessage(content='How will the weather be in munich today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_zxwUrmJCcaPFTWAumbfy6IJZ', 'function': {'arguments': '{"city":"Munich"}', 'name': 'fake_weather_api'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 90, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DQH1apwMtNYAuqIND8kMO8DfHpdFz', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--77c5263e-ea49-4446-9347-412ae4181796-0', tool_calls=[{'name': 'fake_weather_api', 'args': {'city': 'Munich'}, 'id': 'call_zxwUrmJCcaPFTWAumbfy6IJZ', 'type'

In [13]:
llm_with_tools.invoke(messages)

AIMessage(content='The weather in Munich today is sunny with a temperature of 22°C.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 120, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DQH1x0a135ZK50uiTT6woijMtuFNl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--a30b5f09-ed09-482e-b41b-52b8223d4c35-0', usage_metadata={'input_tokens': 120, 'output_tokens': 16, 'total_tokens': 136, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})